# Pré-calcul des features (Colab GPU)

Ce notebook génère les fichiers lourds sur Colab (GPU) pour les utiliser ensuite en local dans les notebooks 04 et 05.

**Fichiers générés :**
- `sbert_train.npy` / `sbert_test.npy` — Embeddings SBERT
- `tfidf_train.npz` / `tfidf_test.npz` + `tfidf_vectorizer.pkl`
- `bow_train.npz` / `bow_test.npz` + `bow_vectorizer.pkl`
- `train_test_split.pkl` — Splits train/test pour cohérence

In [ ]:
!pip install -q sentence-transformers pandas scikit-learn

# Récupération des fichiers via Google Drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os, sys

DRIVE_DIR = '/content/drive/MyDrive/yelp'

print('Fichiers dans le dossier yelp :')
for f in os.listdir(DRIVE_DIR):
    print(f'  {f}')

sys.path.insert(0, DRIVE_DIR)

Mounted at /content/drive
Fichiers dans le dossier yelp :
  yelp_academic_reviews4students.jsonl
  yelp_academic_dataset_business.json
  yelp_academic_dataset_user4students.jsonl
  utils.py
  colab_precompute.ipynb


In [ ]:
import pandas as pd
import numpy as np
import pickle
import scipy.sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer
from src.utils import preprocess_dataframe, load_json_lines

RANDOM_STATE = 42
OUTPUT_DIR = os.path.join(DRIVE_DIR, 'precomputed_features')
os.makedirs(OUTPUT_DIR, exist_ok=True)

Imports OK


In [ ]:
data_file = DRIVE_DIR + '/yelp_academic_reviews4students.jsonl'

df = load_json_lines(data_file)
print(f'Dataset original : {len(df)} reviews')

df = preprocess_dataframe(
    df,
    text_column='text',
    rating_column='stars',
    lowercase=True,
    remove_punctuation=False,
    remove_numbers=False,
    add_polarity=True,
    add_rating=True,
    min_words=10,
    truncate=False
)

print(f'Dataset apres preprocessing : {len(df)} reviews')

Fichier trouve : /content/drive/MyDrive/yelp/yelp_academic_reviews4students.jsonl
  Chargé 100,000 lignes...
  Chargé 200,000 lignes...
  Chargé 300,000 lignes...
  Chargé 400,000 lignes...
  Chargé 500,000 lignes...
  Chargé 600,000 lignes...
  Chargé 700,000 lignes...
  Chargé 800,000 lignes...
  Chargé 900,000 lignes...
  Chargé 1,000,000 lignes...
Dataset original : 1000000 reviews
Dataset apres preprocessing : 997911 reviews


In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text_clean'],
    df[['polarite', 'rating']],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['rating']
)

y_train_pol = y_train['polarite']
y_test_pol = y_test['polarite']
y_train_rating = y_train['rating']
y_test_rating = y_test['rating']

print(f'Train : {len(X_train_text)} | Test : {len(X_test_text)}')

# Sauvegarder les splits
split_data = {
    'X_train': X_train_text,
    'X_test': X_test_text,
    'y_train_pol': y_train_pol,
    'y_test_pol': y_test_pol,
    'y_train_rating': y_train_rating,
    'y_test_rating': y_test_rating,
    'train_index': X_train_text.index.tolist(),
    'test_index': X_test_text.index.tolist()
}

with open(os.path.join(OUTPUT_DIR, 'train_test_split.pkl'), 'wb') as f:
    pickle.dump(split_data, f)

Train : 798328 | Test : 199583
Splits sauvegardes


In [7]:
# 7. BOW Vectorization
bow_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    stop_words='english'
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

scipy.sparse.save_npz(os.path.join(OUTPUT_DIR, 'bow_train.npz'), X_train_bow)
scipy.sparse.save_npz(os.path.join(OUTPUT_DIR, 'bow_test.npz'), X_test_bow)
with open(os.path.join(OUTPUT_DIR, 'bow_vectorizer.pkl'), 'wb') as f:
    pickle.dump(bow_vectorizer, f)

print(f'BOW train : {X_train_bow.shape}')
print(f'BOW test  : {X_test_bow.shape}')
print('BOW sauvegarde')

BOW train : (798328, 5000)
BOW test  : (199583, 5000)
BOW sauvegarde


In [8]:
# 8. TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

scipy.sparse.save_npz(os.path.join(OUTPUT_DIR, 'tfidf_train.npz'), X_train_tfidf)
scipy.sparse.save_npz(os.path.join(OUTPUT_DIR, 'tfidf_test.npz'), X_test_tfidf)
with open(os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print(f'TF-IDF train : {X_train_tfidf.shape}')
print(f'TF-IDF test  : {X_test_tfidf.shape}')
print('TF-IDF sauvegarde')

TF-IDF train : (798328, 5000)
TF-IDF test  : (199583, 5000)
TF-IDF sauvegarde


In [ ]:
# 9. Embeddings SBERT
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
print(f'Modele charge sur : {model.device}')
print(f'Dimension : {model.get_sentence_embedding_dimension()}')

X_train_emb = model.encode(
    X_train.tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_emb = model.encode(
    X_test.tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(os.path.join(OUTPUT_DIR, 'sbert_train.npy'), X_train_emb)
np.save(os.path.join(OUTPUT_DIR, 'sbert_test.npy'), X_test_emb)

print(f'\nEmbeddings train : {X_train_emb.shape}')
print(f'Embeddings test  : {X_test_emb.shape}')
print('Embeddings sauvegardes')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Modele charge sur : cuda:0
Dimension : 384

--- Encodage du train set ---


Batches:   0%|          | 0/3119 [00:00<?, ?it/s]


--- Encodage du test set ---


Batches:   0%|          | 0/780 [00:00<?, ?it/s]


Embeddings train : (798328, 384)
Embeddings test  : (199583, 384)
Embeddings sauvegardes


In [ ]:
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    if size > 1024 * 1024:
        print(f'  {f:35s} {size / 1024 / 1024:.1f} MB')
    else:
        print(f'  {f:35s} {size / 1024:.1f} KB')

print(f'\nTout est sauvegarde dans : {OUTPUT_DIR}')

 FICHIERS GENERES
  bow_test.npz                        13.4 MB
  bow_train.npz                       56.3 MB
  bow_vectorizer.pkl                  138.8 KB
  sbert_test.npy                      292.4 MB
  sbert_train.npy                     1169.4 MB
  tfidf_test.npz                      67.7 MB
  tfidf_train.npz                     275.3 MB
  tfidf_vectorizer.pkl                183.6 KB
  train_test_split.pkl                606.4 MB

Tout est sauvegarde dans : /content/drive/MyDrive/yelp/precomputed_features
Recupere le dossier precomputed_features depuis ton Google Drive.
